In [1]:
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install opencv-python pandas matplotlib supervision ultralytics tqdm

In [2]:
import cv2
print("OpenCV version:", cv2.__version__)

OpenCV version: 4.11.0


In [3]:
# ============================================================
# PHASE 2 — CELL 1
# Environment + Imports
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import sys
import time
import shutil
import traceback
import platform

import cv2
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from ultralytics import YOLO


# ============================================================
# Basic Environment Information
# ============================================================

PYTHON_VERSION = sys.version.split()[0]

print("=" * 70)
print("PHASE 2 — FACE DETECTION")
print("=" * 70)

print(f"Python : {PYTHON_VERSION}")
print(f"OS     : {platform.system()} {platform.release()}")
print(f"OpenCV : {cv2.__version__}")
print(f"PyTorch: {torch.__version__}")


# ============================================================
# CUDA Validation
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "\nCUDA is not available.\n"
        "Phase 2 requires an NVIDIA CUDA-enabled PyTorch installation.\n"
        f"PyTorch version: {torch.__version__}\n"
    )


GPU_COUNT = torch.cuda.device_count()

if GPU_COUNT <= 0:

    raise RuntimeError(
        "CUDA is available but no CUDA device was detected."
    )


GPU_INDEX = 0

if GPU_INDEX >= GPU_COUNT:

    raise RuntimeError(
        f"GPU_INDEX={GPU_INDEX} is invalid. "
        f"Available GPUs: {GPU_COUNT}"
    )


DEVICE = f"cuda:{GPU_INDEX}"

GPU_NAME = torch.cuda.get_device_name(
    GPU_INDEX
)


# ============================================================
# CUDA Information
# ============================================================

print()
print("CUDA")
print("-" * 70)

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"CUDA devices   : {GPU_COUNT}")
print(f"Device         : {DEVICE}")
print(f"GPU            : {GPU_NAME}")

if torch.version.cuda:

    print(
        f"PyTorch CUDA   : {torch.version.cuda}"
    )

else:

    print(
        "PyTorch CUDA   : None"
    )


# ============================================================
# CUDA Warm-up
# ============================================================

try:

    torch.cuda.set_device(
        GPU_INDEX
    )

    torch.cuda.empty_cache()

    torch.cuda.synchronize()

    print()
    print("CUDA initialization: PASS")

except Exception as e:

    raise RuntimeError(
        f"CUDA initialization failed: {e}"
    )


print("=" * 70)

PHASE 2 — FACE DETECTION
Python : 3.11.9
OS     : Windows 10
OpenCV : 4.11.0
PyTorch: 2.13.0+cu126

CUDA
----------------------------------------------------------------------
CUDA available : True
CUDA devices   : 1
Device         : cuda:0
GPU            : NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch CUDA   : 12.6

CUDA initialization: PASS


In [4]:
# ============================================================
# PHASE 2 — CELL 2
# Project Paths + Production Configuration
# ============================================================


# ============================================================
# Project Root Detection
# ============================================================

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:

    current = start_path

    while True:

        raw_video_dir = (
            current
            / "dataset"
            / "raw_videos"
        )

        models_dir = (
            current
            / "models"
        )

        output_dir = (
            current
            / "output"
        )

        if (
            raw_video_dir.exists()
            or models_dir.exists()
            or output_dir.exists()
        ):

            return current

        if current == current.parent:

            break

        current = current.parent

    # Fallback
    return start_path


PROJECT_DIR = find_project_root(
    CURRENT_DIR
)


# ============================================================
# Main Directories
# ============================================================

DATASET_DIR = (
    PROJECT_DIR
    / "dataset"
)

RAW_VIDEO_DIR = (
    DATASET_DIR
    / "raw_videos"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "output"
)

MODEL_PATH = (
    PROJECT_DIR
    / "models"
    / "yolo11n.pt"
)


# ============================================================
# Supported Video Formats
# ============================================================

SUPPORTED_VIDEO_EXTENSIONS = {

    ".webm",
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".mpeg",
    ".mpg"

}


# ============================================================
# YOLO Configuration
# ============================================================

CONFIDENCE_THRESHOLD = 0.50

IOU_THRESHOLD = 0.45

DEVICE = GPU_INDEX


# ============================================================
# Processing Mode
# ============================================================

# False:
#   Skip completed videos.
#
# True:
#   Reprocess every selected video.
#
FORCE_RERUN = False


# ============================================================
# Resume Configuration
# ============================================================

# True:
#   Continue incomplete processing.
#
# False:
#   Reset incomplete processing.
#
RESUME_ENABLED = True


# ============================================================
# Detection Configuration
# ============================================================

# YOLO inference batch size.
#
# Start conservatively.
#
BATCH_SIZE = 16


# ============================================================
# Output Configuration
# ============================================================

SAVE_DETECTION_CSV = True

SAVE_METADATA = True

SAVE_PROCESSING_STATE = True


# ============================================================
# CSV Schema
# ============================================================

DETECTION_COLUMNS = [

    "video_id",
    "frame_index",
    "timestamp",
    "face_detected",
    "confidence",
    "x1",
    "y1",
    "x2",
    "y2",

]


# ============================================================
# File Names
# ============================================================

DETECTION_CSV_NAME = (
    "detection.csv"
)

STATE_FILE_NAME = (
    "processing_state.json"
)

METADATA_FILE_NAME = (
    "processing_metadata.json"
)

LOCK_FILE_NAME = (
    ".phase2.lock"
)


# ============================================================
# Backup Configuration
# ============================================================

BACKUP_DIR_NAME = (
    "backups"
)


# ============================================================
# Logging
# ============================================================

LOG_DIR_NAME = (
    "phase2_logs"
)

RUN_LOG_NAME = (
    "run.log"
)


# ============================================================
# Print Configuration
# ============================================================

print("=" * 70)
print("PHASE 2 — PRODUCTION CONFIGURATION")
print("=" * 70)

print(f"Project       : {PROJECT_DIR}")
print(f"Raw videos    : {RAW_VIDEO_DIR}")
print(f"Output        : {OUTPUT_DIR}")
print(f"YOLO model    : {MODEL_PATH}")
print(f"Device        : CUDA:{GPU_INDEX}")
print(f"GPU           : {GPU_NAME}")
print(f"Confidence    : {CONFIDENCE_THRESHOLD}")
print(f"IoU           : {IOU_THRESHOLD}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Force rerun   : {FORCE_RERUN}")
print(f"Resume        : {RESUME_ENABLED}")

print("=" * 70)


# ============================================================
# Directory Validation
# ============================================================

if not RAW_VIDEO_DIR.exists():

    raise FileNotFoundError(
        f"\nRaw video directory not found:\n"
        f"{RAW_VIDEO_DIR}"
    )


if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"\nYOLO model not found:\n"
        f"{MODEL_PATH}"
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Model Validation
# ============================================================

print()
print("Loading YOLO model...")

FACE_MODEL = YOLO(
    str(MODEL_PATH)
)

try:

    FACE_MODEL.to(
        DEVICE
    )

except Exception as e:

    raise RuntimeError(
        f"Failed to move YOLO model to {DEVICE}: {e}"
    )


print(
    f"YOLO model loaded on {DEVICE}"
)

print()
print("Configuration validation: PASS")

PHASE 2 — PRODUCTION CONFIGURATION
Project       : C:\LipReadingSSL
Raw videos    : C:\LipReadingSSL\dataset\raw_videos
Output        : C:\LipReadingSSL\output
YOLO model    : C:\LipReadingSSL\models\yolo11n.pt
Device        : CUDA:0
GPU           : NVIDIA GeForce RTX 4050 Laptop GPU
Confidence    : 0.5
IoU           : 0.45
Batch size    : 16
Force rerun   : False
Resume        : True

Loading YOLO model...
YOLO model loaded on 0

Configuration validation: PASS


In [5]:
# ============================================================
# PHASE 2 — CELL 3
# Utility Functions
# ============================================================


# ============================================================
# Timestamp
# ============================================================

def utc_now_iso():

    return datetime.utcnow().isoformat(
        timespec="seconds"
    ) + "Z"


# ============================================================
# JSON Helpers
# ============================================================

def write_json(
    path: Path,
    data: dict
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    # Windows-safe replacement
    try:

        if path.exists():

            path.unlink()

        temp_path.replace(
            path
        )

    except PermissionError:

        # Fallback if Windows temporarily
        # locks the target file.

        shutil.copy2(
            temp_path,
            path
        )

        temp_path.unlink(
            missing_ok=True
        )


# ============================================================
# SHA256
# ============================================================

def calculate_file_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024
):

    file_path = Path(
        file_path
    )

    sha256 = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:

                break

            sha256.update(
                chunk
            )

    return sha256.hexdigest()


# ============================================================
# Video ID
# ============================================================

def get_video_id(
    video_path: Path
):

    return video_path.stem


# ============================================================
# Video Output Directory
# ============================================================

def get_video_output_dir(
    video_path: Path
):

    video_id = get_video_id(
        video_path
    )

    return (
        OUTPUT_DIR
        / video_id
    )


# ============================================================
# Video Output Paths
# ============================================================

def get_video_paths(
    video_path: Path
):

    video_output_dir = (
        get_video_output_dir(
            video_path
        )
    )

    log_dir = (
        video_output_dir
        / LOG_DIR_NAME
    )

    backup_dir = (
        video_output_dir
        / BACKUP_DIR_NAME
    )

    return {

        "video_id":
            get_video_id(video_path),

        "output_dir":
            video_output_dir,

        "detection_csv":
            video_output_dir
            / DETECTION_CSV_NAME,

        "state":
            video_output_dir
            / STATE_FILE_NAME,

        "metadata":
            video_output_dir
            / METADATA_FILE_NAME,

        "lock":
            video_output_dir
            / LOCK_FILE_NAME,

        "log_dir":
            log_dir,

        "run_log":
            log_dir
            / RUN_LOG_NAME,

        "backup_dir":
            backup_dir,

    }


# ============================================================
# Video Information
# ============================================================

def get_video_info(
    video_path: Path
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    duration = (
        frame_count / fps
        if fps > 0
        else 0
    )

    cap.release()

    return {

        "frame_count":
            frame_count,

        "fps":
            fps,

        "width":
            width,

        "height":
            height,

        "duration":
            duration,

    }


# ============================================================
# Video Fingerprint
# ============================================================

def create_video_fingerprint(
    video_path: Path
):

    stat = video_path.stat()

    return {

        "sha256":
            calculate_file_sha256(
                video_path
            ),

        "filename":
            video_path.name,

        "size_bytes":
            stat.st_size,

        "modified_time":
            stat.st_mtime,

    }


# ============================================================
# CSV Schema
# ============================================================

def normalize_detection_dataframe(
    df: pd.DataFrame
):

    df = df.copy()

    for column in DETECTION_COLUMNS:

        if column not in df.columns:

            df[column] = np.nan

    return df[
        DETECTION_COLUMNS
    ]


# ============================================================
# Detection Record
# ============================================================

def create_detection_record(
    video_id,
    frame_index,
    timestamp,
    face_detected,
    confidence=None,
    x1=None,
    y1=None,
    x2=None,
    y2=None
):

    return {

        "video_id":
            video_id,

        "frame_index":
            int(frame_index),

        "timestamp":
            float(timestamp),

        "face_detected":
            int(face_detected),

        "confidence":
            confidence,

        "x1":
            x1,

        "y1":
            y1,

        "x2":
            x2,

        "y2":
            y2,

    }


print(
    "Utility functions loaded: PASS"
)

Utility functions loaded: PASS


In [6]:
# ============================================================
# PHASE 2 — CELL 4
# CSV Validation + State + Backup
# ============================================================

import re


# ============================================================
# Required CSV Schema
# ============================================================

EXPECTED_DETECTION_COLUMNS = set(
    DETECTION_COLUMNS
)


# ============================================================
# Read JSON Safely
# ============================================================

def read_json_safe(
    path: Path
):

    path = Path(path)

    if not path.exists():

        return None

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    except Exception:

        return None


# ============================================================
# Safe State Writer
# ============================================================

def write_processing_state(
    video_path: Path,
    state: dict
):

    paths = get_video_paths(
        video_path
    )

    state_path = paths["state"]

    state_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    state = dict(state)

    state["updated_at"] = (
        utc_now_iso()
    )

    temp_path = state_path.with_name(
        f"{state_path.stem}.{os.getpid()}.tmp"
    )

    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            indent=2,
            ensure_ascii=False
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    # --------------------------------------------------------
    # Windows-safe replacement with retry
    # --------------------------------------------------------

    last_error = None

    for attempt in range(5):

        try:

            os.replace(
                temp_path,
                state_path
            )

            return

        except PermissionError as e:

            last_error = e

            time.sleep(
                0.25 * (attempt + 1)
            )

    # --------------------------------------------------------
    # Last fallback
    # --------------------------------------------------------

    try:

        with open(
            state_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                state,
                f,
                indent=2,
                ensure_ascii=False
            )

    except Exception:

        temp_path.unlink(
            missing_ok=True
        )

        raise last_error


# ============================================================
# Backup Existing File
# ============================================================

def backup_file(
    file_path: Path,
    video_path: Path,
    reason: str
):

    file_path = Path(
        file_path
    )

    if not file_path.exists():

        return None

    paths = get_video_paths(
        video_path
    )

    backup_dir = (
        paths["backup_dir"]
    )

    backup_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    safe_reason = re.sub(
        r"[^a-zA-Z0-9_-]+",
        "_",
        str(reason)
    )

    backup_name = (
        f"{file_path.stem}"
        f"_{timestamp}"
        f"_{safe_reason}"
        f"{file_path.suffix}"
    )

    backup_path = (
        backup_dir
        / backup_name
    )

    shutil.copy2(
        file_path,
        backup_path
    )

    return backup_path


# ============================================================
# Validate Detection CSV
# ============================================================

def validate_detection_csv(
    video_path: Path,
    csv_path: Path = None
):

    video_path = Path(
        video_path
    ).resolve()

    paths = get_video_paths(
        video_path
    )

    if csv_path is None:

        csv_path = (
            paths["detection_csv"]
        )

    csv_path = Path(
        csv_path
    ).resolve()

    result = {

        "valid": False,

        "reason": None,

        "rows": 0,

        "unique_frames": 0,

        "expected_frames": 0,

        "first_frame": None,

        "last_frame": None,

        "missing_frames": 0,

        "columns": [],

        "missing_columns": [],

    }

    # --------------------------------------------------------
    # CSV exists
    # --------------------------------------------------------

    if not csv_path.exists():

        result["reason"] = (
            "missing_detection_csv"
        )

        return result

    # --------------------------------------------------------
    # Read video info
    # --------------------------------------------------------

    try:

        video_info = get_video_info(
            video_path
        )

        expected_frames = int(
            video_info["frame_count"]
        )

        result["expected_frames"] = (
            expected_frames
        )

    except Exception as e:

        result["reason"] = (
            f"video_info_error:{e}"
        )

        return result

    # --------------------------------------------------------
    # Read CSV
    # --------------------------------------------------------

    try:

        df = pd.read_csv(
            csv_path
        )

    except Exception as e:

        result["reason"] = (
            f"csv_read_error:{e}"
        )

        return result

    result["rows"] = len(df)

    result["columns"] = (
        list(df.columns)
    )

    # --------------------------------------------------------
    # Schema
    # --------------------------------------------------------

    missing_columns = [

        col

        for col in DETECTION_COLUMNS

        if col not in df.columns

    ]

    result["missing_columns"] = (
        missing_columns
    )

    if missing_columns:

        result["reason"] = (
            "missing_columns:"
            +
            ",".join(
                missing_columns
            )
        )

        return result

    # --------------------------------------------------------
    # Empty CSV
    # --------------------------------------------------------

    if len(df) == 0:

        result["reason"] = (
            "empty_detection_csv"
        )

        return result

    # --------------------------------------------------------
    # Frame index
    # --------------------------------------------------------

    frame_index = pd.to_numeric(
        df["frame_index"],
        errors="coerce"
    )

    if frame_index.isna().any():

        result["reason"] = (
            "invalid_frame_index"
        )

        return result

    frame_index = (
        frame_index
        .astype(int)
    )

    if (frame_index < 0).any():

        result["reason"] = (
            "negative_frame_index"
        )

        return result

    # --------------------------------------------------------
    # Frame range
    # --------------------------------------------------------

    result["first_frame"] = int(
        frame_index.min()
    )

    result["last_frame"] = int(
        frame_index.max()
    )

    result["unique_frames"] = int(
        frame_index.nunique()
    )

    if (
        result["first_frame"] != 0
    ):

        result["reason"] = (
            f"first_frame_not_zero:"
            f"{result['first_frame']}"
        )

        return result

    if (
        result["last_frame"]
        != expected_frames - 1
    ):

        result["reason"] = (
            f"last_frame_mismatch:"
            f"{result['last_frame']}"
            f"/"
            f"{expected_frames - 1}"
        )

        return result

    # --------------------------------------------------------
    # Missing frame check
    # --------------------------------------------------------

    actual_frames = set(
        frame_index.tolist()
    )

    expected_frame_set = set(
        range(
            expected_frames
        )
    )

    missing_frames = (
        expected_frame_set
        - actual_frames
    )

    result["missing_frames"] = len(
        missing_frames
    )

    if missing_frames:

        result["reason"] = (
            f"missing_frames:"
            f"{len(missing_frames)}"
        )

        return result

    # --------------------------------------------------------
    # Video ID check
    # --------------------------------------------------------

    expected_video_id = (
        get_video_id(
            video_path
        )
    )

    actual_video_ids = (
        df["video_id"]
        .dropna()
        .astype(str)
        .unique()
    )

    if len(actual_video_ids) != 1:

        result["reason"] = (
            "invalid_video_id_count"
        )

        return result

    if (
        actual_video_ids[0]
        != expected_video_id
    ):

        result["reason"] = (
            "video_id_mismatch:"
            f"{actual_video_ids[0]}"
        )

        return result

    # --------------------------------------------------------
    # Face detected validation
    # --------------------------------------------------------

    face_detected = pd.to_numeric(
        df["face_detected"],
        errors="coerce"
    )

    if face_detected.isna().any():

        result["reason"] = (
            "invalid_face_detected"
        )

        return result

    if not set(
        face_detected.astype(int)
    ).issubset({0, 1}):

        result["reason"] = (
            "invalid_face_detected_values"
        )

        return result

    # --------------------------------------------------------
    # PASS
    # --------------------------------------------------------

    result["valid"] = True

    result["reason"] = (
        "complete"
    )

    return result


print(
    "Cell 4 loaded: CSV validation + state + backup"
)

Cell 4 loaded: CSV validation + state + backup


In [7]:
# ============================================================
# PHASE 2 — CELL 5
# Lock + Decision Engine
# ============================================================


# ============================================================
# Process Lock
# ============================================================

class VideoProcessLock:

    def __init__(
        self,
        video_path: Path
    ):

        self.video_path = (
            Path(video_path)
            .resolve()
        )

        self.paths = get_video_paths(
            self.video_path
        )

        self.lock_path = (
            self.paths["lock"]
        )

        self.acquired = False


    # --------------------------------------------------------
    # Acquire
    # --------------------------------------------------------

    def acquire(self):

        self.lock_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if self.lock_path.exists():

            lock_data = read_json_safe(
                self.lock_path
            )

            pid = (
                lock_data.get(
                    "pid"
                )
                if lock_data
                else None
            )

            started_at = (
                lock_data.get(
                    "started_at"
                )
                if lock_data
                else None
            )

            raise RuntimeError(
                "\nVideo is already locked.\n"
                f"Video : {self.video_path.name}\n"
                f"Lock  : {self.lock_path}\n"
                f"PID   : {pid}\n"
                f"Since : {started_at}\n"
                "\n"
                "If you are certain no other Phase 2 "
                "process is running, remove the .phase2.lock file."
            )

        lock_data = {

            "pid":
                os.getpid(),

            "video":
                self.video_path.name,

            "video_id":
                get_video_id(
                    self.video_path
                ),

            "started_at":
                utc_now_iso(),

        }

        try:

            # Exclusive creation.
            # If another process creates the file
            # at the same time, this fails.

            fd = os.open(
                str(self.lock_path),
                os.O_CREAT
                | os.O_EXCL
                | os.O_WRONLY
            )

            with os.fdopen(
                fd,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    lock_data,
                    f,
                    indent=2
                )

            self.acquired = True

        except FileExistsError:

            raise RuntimeError(
                f"Concurrent execution detected for "
                f"{self.video_path.name}"
            )


    # --------------------------------------------------------
    # Release
    # --------------------------------------------------------

    def release(self):

        if not self.acquired:

            return

        try:

            if self.lock_path.exists():

                self.lock_path.unlink()

        finally:

            self.acquired = False


    # --------------------------------------------------------
    # Context Manager
    # --------------------------------------------------------

    def __enter__(self):

        self.acquire()

        return self


    def __exit__(
        self,
        exc_type,
        exc_value,
        traceback_obj
    ):

        self.release()


# ============================================================
# Load Existing State
# ============================================================

def load_processing_state(
    video_path: Path
):

    paths = get_video_paths(
        video_path
    )

    return read_json_safe(
        paths["state"]
    )


# ============================================================
# Check State Fingerprint
# ============================================================

def state_matches_video(
    state: dict,
    fingerprint: dict
):

    if not state:

        return False

    state_fingerprint = (
        state.get(
            "video_fingerprint"
        )
    )

    if not state_fingerprint:

        return False

    return (
        state_fingerprint.get("sha256")
        ==
        fingerprint.get("sha256")
    )


# ============================================================
# Decision Engine
# ============================================================

def determine_video_action(
    video_path: Path
):

    video_path = (
        Path(video_path)
        .resolve()
    )

    paths = get_video_paths(
        video_path
    )

    fingerprint = (
        create_video_fingerprint(
            video_path
        )
    )

    validation = (
        validate_detection_csv(
            video_path
        )
    )

    state = (
        load_processing_state(
            video_path
        )
    )

    # ========================================================
    # FORCE RERUN
    # ========================================================

    if FORCE_RERUN:

        return {

            "action":
                "RESET",

            "reason":
                "force_rerun",

            "validation":
                validation,

            "state":
                state,

            "fingerprint":
                fingerprint,

        }


    # ========================================================
    # Completed CSV + Matching State
    # ========================================================

    if validation["valid"]:

        if state_matches_video(
            state,
            fingerprint
        ):

            if (
                state.get("status")
                == "COMPLETED"
            ):

                return {

                    "action":
                        "SKIP",

                    "reason":
                        "completed",

                    "validation":
                        validation,

                    "state":
                        state,

                    "fingerprint":
                        fingerprint,

                }

        # ----------------------------------------------------
        # CSV is complete even if state is missing/old.
        # Do NOT rerun YOLO unnecessarily.
        # ----------------------------------------------------

        return {

            "action":
                "SKIP",

            "reason":
                "valid_detection_csv",

            "validation":
                validation,

            "state":
                state,

            "fingerprint":
                fingerprint,

        }


    # ========================================================
    # Incomplete CSV
    # ========================================================

    if (
        validation["reason"]
        == "missing_detection_csv"
    ):

        return {

            "action":
                "RUN",

            "reason":
                "no_existing_csv",

            "validation":
                validation,

            "state":
                state,

            "fingerprint":
                fingerprint,

        }


    # ========================================================
    # Existing CSV + Resume
    # ========================================================

    if (
        RESUME_ENABLED
        and
        validation["rows"] > 0
        and
        state_matches_video(
            state,
            fingerprint
        )
    ):

        return {

            "action":
                "RESUME",

            "reason":
                validation["reason"],

            "validation":
                validation,

            "state":
                state,

            "fingerprint":
                fingerprint,

        }


    # ========================================================
    # Existing CSV but unsafe to resume
    # ========================================================

    return {

        "action":
            "RESET",

        "reason":
            validation["reason"],

        "validation":
            validation,

        "state":
            state,

        "fingerprint":
            fingerprint,

    }


print(
    "Cell 5 loaded: lock + decision engine"
)

Cell 5 loaded: lock + decision engine


In [8]:
# ============================================================
# PHASE 2 — CELL 6
# Production Validation + Safe Resume Decision
#
# Compatible with:
#   Cell 7 — SKIP_FRAME
#   Cell 8 — Production Batch Runner
#
# IMPORTANT:
#   Missing frames are NOT automatically an error.
#
#   A video can be:
#
#       COMPLETED
#       COMPLETED_WITH_SKIPS
#
# if the final frame has been processed or explicitly skipped.
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# Required Detection CSV Schema
# ============================================================

REQUIRED_DETECTION_COLUMNS = [

    "video_id",

    "frame_index",

    "timestamp",

    "face_detected",

    "track_id",

]


# ============================================================
# Read Processing State
# ============================================================

def _read_state_safe(
    video_path
):

    try:

        state = read_processing_state(
            video_path
        )

        if isinstance(
            state,
            dict
        ):

            return state

    except Exception:

        pass

    return {}


# ============================================================
# Extract Skipped Frames From State
# ============================================================

def _get_skipped_frames_from_state(
    state
):

    skipped = state.get(
        "skipped_frames",
        []
    )

    if not isinstance(
        skipped,
        (list, tuple, set)
    ):

        return []

    clean = []

    for frame in skipped:

        try:

            clean.append(
                int(frame)
            )

        except Exception:

            continue

    return sorted(
        set(clean)
    )


# ============================================================
# Validate Detection CSV
# ============================================================

def validate_detection_csv(
    video_path
):

    video_path = (
        Path(
            video_path
        )
        .resolve()
    )

    paths = get_video_paths(
        video_path
    )

    csv_path = (
        paths[
            "detection_csv"
        ]
    )

    result = {

        "valid":
            False,

        "reason":
            None,

        "rows":
            0,

        "first_frame":
            None,

        "last_frame":
            None,

        "expected_frames":
            None,

        "expected_last_frame":
            None,

        "processed_frames":
            0,

        "missing_frames":
            [],

        "skipped_frames":
            [],

        "unresolved_frames":
            [],

        "missing_columns":
            [],

    }

    # ========================================================
    # 1. CSV DOES NOT EXIST
    # ========================================================

    if not csv_path.exists():

        result[
            "reason"
        ] = (
            "missing_detection_csv"
        )

        return result


    # ========================================================
    # 2. READ CSV
    # ========================================================

    try:

        df = pd.read_csv(
            csv_path
        )

    except Exception as e:

        result[
            "reason"
        ] = (

            "csv_read_error:"
            +
            type(e).__name__

        )

        return result


    result[
        "rows"
    ] = len(df)


    # ========================================================
    # 3. VALIDATE SCHEMA
    # ========================================================

    missing_columns = [

        column

        for column
        in REQUIRED_DETECTION_COLUMNS

        if column
        not in df.columns

    ]

    result[
        "missing_columns"
    ] = missing_columns


    if missing_columns:

        result[
            "reason"
        ] = (

            "missing_columns:"
            +
            ",".join(
                missing_columns
            )

        )

        return result


    # ========================================================
    # 4. EMPTY CSV
    # ========================================================

    if len(df) == 0:

        result[
            "reason"
        ] = (
            "empty_detection_csv"
        )

        return result


    # ========================================================
    # 5. VALIDATE FRAME INDEX
    # ========================================================

    frame_index = pd.to_numeric(

        df[
            "frame_index"
        ],

        errors="coerce"

    )


    if frame_index.isna().any():

        result[
            "reason"
        ] = (
            "invalid_frame_index"
        )

        return result


    frame_index = (

        frame_index
        .astype(int)

    )


    unique_frames = sorted(

        set(
            frame_index.tolist()
        )

    )


    if not unique_frames:

        result[
            "reason"
        ] = (
            "no_valid_frames"
        )

        return result


    result[
        "first_frame"
    ] = unique_frames[0]

    result[
        "last_frame"
    ] = unique_frames[-1]

    result[
        "processed_frames"
    ] = len(
        unique_frames
    )


    # ========================================================
    # 6. GET VIDEO FRAME COUNT
    # ========================================================

    try:

        video_info = (
            get_video_info(
                video_path
            )
        )

    except Exception as e:

        result[
            "reason"
        ] = (

            "video_info_error:"
            +
            type(e).__name__

        )

        return result


    total_frames = int(

        video_info[
            "frame_count"
        ]

    )


    expected_last = (
        total_frames - 1
    )


    result[
        "expected_frames"
    ] = total_frames


    result[
        "expected_last_frame"
    ] = expected_last


    # ========================================================
    # 7. INVALID FRAME RANGE
    # ========================================================

    invalid_frames = [

        frame

        for frame
        in unique_frames

        if (

            frame < 0

            or

            frame > expected_last

        )

    ]


    if invalid_frames:

        result[
            "reason"
        ] = (

            "invalid_frame_range:"
            +
            str(
                invalid_frames[
                    :10
                ]
            )

        )

        return result


    # ========================================================
    # 8. READ PROCESSING STATE
    # ========================================================

    state = _read_state_safe(
        video_path
    )


    skipped_frames = (
        _get_skipped_frames_from_state(
            state
        )
    )


    result[
        "skipped_frames"
    ] = skipped_frames


    # ========================================================
    # 9. FIND MISSING FRAMES
    #
    # Missing frame is NOT automatically an error.
    # ========================================================

    actual_set = set(
        unique_frames
    )


    expected_set = set(

        range(
            0,
            total_frames
        )

    )


    missing_frames = sorted(

        expected_set
        -
        actual_set

    )


    result[
        "missing_frames"
    ] = missing_frames


    # ========================================================
    # 10. DETERMINE UNRESOLVED FRAMES
    #
    # A frame is unresolved when:
    #
    #   missing from CSV
    #   AND
    #   not recorded as SKIP_FRAME
    #
    # These frames still need processing.
    # ========================================================

    skipped_set = set(
        skipped_frames
    )


    unresolved_frames = sorted(

        set(
            missing_frames
        )
        -
        skipped_set

    )


    result[
        "unresolved_frames"
    ] = (
        unresolved_frames
    )


    # ========================================================
    # 11. FINAL FRAME STATUS
    # ========================================================

    final_frame_in_csv = (

        expected_last
        in actual_set

    )


    final_frame_skipped = (

        expected_last
        in skipped_set

    )


    final_frame_done = (

        final_frame_in_csv

        or

        final_frame_skipped

    )


    # ========================================================
    # 12. COMPLETED
    #
    # If final frame exists OR was skipped,
    # processing reached the end of the video.
    # ========================================================

    if final_frame_done:

        if skipped_frames:

            result[
                "valid"
            ] = True

            result[
                "reason"
            ] = (
                "valid_with_skips"
            )

        else:

            result[
                "valid"
            ] = True

            result[
                "reason"
            ] = (
                "valid_detection_csv"
            )

        return result


    # ========================================================
    # 13. NOT COMPLETED YET
    #
    # There are unresolved frames and the final frame
    # has not been processed/skipped.
    # ========================================================

    if unresolved_frames:

        first_unresolved = (
            unresolved_frames[0]
        )

        # ----------------------------------------------------
        # Check whether all unresolved frames are at the tail
        # ----------------------------------------------------

        expected_tail = list(

            range(

                first_unresolved,

                total_frames

            )

        )


        unresolved_set = set(
            unresolved_frames
        )


        if unresolved_set.issubset(
            set(
                expected_tail
            )
        ):

            result[
                "reason"
            ] = (

                "missing_tail_frames:"
                +
                f"{first_unresolved}-"
                +
                f"{expected_last}"

            )

            return result


        # ----------------------------------------------------
        # Middle gap
        # ----------------------------------------------------

        result[
            "reason"
        ] = (

            "middle_frame_gap:"
            +
            str(
                first_unresolved
            )

        )

        return result


    # ========================================================
    # 14. FALLBACK
    # ========================================================

    result[
        "reason"
    ] = (
        "processing_incomplete"
    )

    return result


# ============================================================
# Determine Processing Action
# ============================================================

def determine_processing_action(
    video_path
):

    validation = (
        validate_detection_csv(
            video_path
        )
    )


    # ========================================================
    # FORCE RERUN
    # ========================================================

    if FORCE_RERUN:

        return {

            "action":
                "RESET",

            "reason":
                "force_rerun",

            "start_frame":
                0,

            "validation":
                validation,

        }


    # ========================================================
    # COMPLETED
    #
    # Important:
    # valid_with_skips is also COMPLETED.
    # ========================================================

    if validation[
        "valid"
    ]:

        return {

            "action":
                "SKIP",

            "reason":
                validation[
                    "reason"
                ],

            "start_frame":
                None,

            "validation":
                validation,

        }


    # ========================================================
    # NO CSV
    # ========================================================

    if validation[
        "reason"
    ] == "missing_detection_csv":

        return {

            "action":
                "RUN",

            "reason":
                "missing_detection_csv",

            "start_frame":
                0,

            "validation":
                validation,

        }


    # ========================================================
    # MISSING TAIL
    #
    # Resume from first unresolved frame.
    # ========================================================

    reason = validation[
        "reason"
    ]


    if (

        isinstance(
            reason,
            str
        )

        and

        reason.startswith(
            "missing_tail_frames:"
        )

    ):

        first_missing = int(

            reason
            .split(":")[1]
            .split("-")[0]

        )


        return {

            "action":
                "RESUME",

            "reason":
                reason,

            "start_frame":
                first_missing,

            "validation":
                validation,

        }


    # ========================================================
    # MIDDLE GAP
    #
    # We do NOT automatically reset.
    #
    # Resume from the first unresolved frame.
    #
    # Cell 7 can SKIP_FRAME if that frame cannot be read.
    # ========================================================

    if (

        isinstance(
            reason,
            str
        )

        and

        reason.startswith(
            "middle_frame_gap:"
        )

    ):

        first_missing = int(

            reason
            .split(":")[1]

        )


        return {

            "action":
                "RESUME",

            "reason":
                reason,

            "start_frame":
                first_missing,

            "validation":
                validation,

        }


    # ========================================================
    # INVALID SCHEMA / CORRUPTED CSV / OTHER
    #
    # Reset is still appropriate here.
    # ========================================================

    return {

        "action":
            "RESET",

        "reason":
            validation[
                "reason"
            ],

        "start_frame":
            0,

        "validation":
            validation,

    }


# ============================================================
# Ready
# ============================================================

print("=" * 70)
print(
    "Cell 6 — SKIP_FRAME Compatible "
    "Validation System Ready"
)
print("=" * 70)

print()
print(
    "Missing frames : ALLOWED"
)

print(
    "Skipped frames : ALLOWED"
)

print(
    "Final frame    : CSV OR SKIP_FRAME"
)

print(
    "Middle gaps    : RESUME"
)

print(
    "Invalid schema : RESET"
)

print("=" * 70)

Cell 6 — SKIP_FRAME Compatible Validation System Ready

Missing frames : ALLOWED
Skipped frames : ALLOWED
Final frame    : CSV OR SKIP_FRAME
Middle gaps    : RESUME
Invalid schema : RESET


In [9]:
# ============================================================
# PHASE 2 — CELL 7
# PRODUCTION PROCESSING ENGINE
#
# VERSION:
#   SKIP_FRAME
#
# Main behavior:
#   - RESET
#   - BACKUP
#   - RESUME
#   - SKIP unreadable frames
#   - .webm support
#   - CUDA support
#   - duplicate frame protection
#   - state tracking
#   - final validation
#
# IMPORTANT:
#   Missing frames do NOT automatically mean FAILED.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import gc
import os
import time

import cv2
import pandas as pd
import torch

from tqdm.auto import tqdm
from ultralytics import YOLO


# ============================================================
# 1. CONSTANTS
# ============================================================

DETECTION_COLUMNS = [
    "video_id",
    "frame_index",
    "timestamp",
    "face_detected",
    "track_id",
]


# ============================================================
# 2. TIME
# ============================================================

def utc_now_iso():

    return datetime.now(
        timezone.utc
    ).isoformat()


# ============================================================
# 3. VIDEO ID
# ============================================================

def get_video_id(video_path):

    return Path(
        video_path
    ).stem


# ============================================================
# 4. VIDEO PATHS
# ============================================================

def get_video_paths(video_path):

    video_path = (
        Path(video_path)
        .resolve()
    )

    video_id = get_video_id(
        video_path
    )

    video_output_dir = (
        OUTPUT_DIR / video_id
    )

    backup_dir = (
        video_output_dir / "backups"
    )

    return {

        "video":
            video_path,

        "video_id":
            video_id,

        "output_dir":
            video_output_dir,

        "detection_csv":
            video_output_dir
            / "detection.csv",

        "state_json":
            video_output_dir
            / "processing_state.json",

        "lock_file":
            video_output_dir
            / ".phase2.lock",

        "backup_dir":
            backup_dir,

    }


# ============================================================
# 5. ENSURE OUTPUT DIRECTORY
# ============================================================

def ensure_video_output_dir(
    video_path
):

    paths = get_video_paths(
        video_path
    )

    paths["output_dir"].mkdir(
        parents=True,
        exist_ok=True
    )

    paths["backup_dir"].mkdir(
        parents=True,
        exist_ok=True
    )

    return paths


# ============================================================
# 6. VIDEO INFORMATION
# ============================================================

def get_video_info(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open video:\n"
            f"{video_path}"
        )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    cap.release()

    if frame_count <= 0:

        raise RuntimeError(
            f"Invalid frame count: "
            f"{frame_count}"
        )

    if fps <= 0:

        fps = 25.0

    return {

        "frame_count":
            frame_count,

        "fps":
            fps,

        "width":
            width,

        "height":
            height,

    }


# ============================================================
# 7. PROCESS LOCK
# ============================================================

class VideoProcessLock:

    def __init__(
        self,
        video_path
    ):

        self.video_path = (
            Path(video_path)
            .resolve()
        )

        self.paths = (
            ensure_video_output_dir(
                self.video_path
            )
        )

        self.lock_file = (
            self.paths["lock_file"]
        )

    def __enter__(self):

        if self.lock_file.exists():

            raise RuntimeError(

                "Concurrent processing "
                "detected.\n"
                f"Lock file already exists:\n"
                f"{self.lock_file}"

            )

        payload = {

            "pid":
                os.getpid(),

            "video":
                str(self.video_path),

            "created_at":
                utc_now_iso(),

        }

        self.lock_file.write_text(

            json.dumps(
                payload,
                indent=2
            ),

            encoding="utf-8"

        )

        return self

    def __exit__(
        self,
        exc_type,
        exc_value,
        traceback
    ):

        try:

            if self.lock_file.exists():

                self.lock_file.unlink()

        except Exception as e:

            print(
                "WARNING: "
                "Could not remove lock:"
            )

            print(e)

        return False


# ============================================================
# 8. BACKUP DETECTION CSV
# ============================================================

def backup_detection_csv(
    video_path,
    reason="manual_backup"
):

    paths = ensure_video_output_dir(
        video_path
    )

    csv_path = (
        paths["detection_csv"]
    )

    if not csv_path.exists():

        return None

    timestamp = (
        datetime.now()
        .strftime(
            "%Y%m%d_%H%M%S"
        )
    )

    safe_reason = (
        str(reason)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
    )

    backup_path = (

        paths["backup_dir"]
        /
        (
            f"detection_"
            f"{timestamp}_"
            f"{safe_reason}.csv"
        )

    )

    shutil.copy2(
        csv_path,
        backup_path
    )

    print()
    print(
        "Backup created:"
    )

    print(
        f"  {backup_path}"
    )

    return backup_path


# ============================================================
# 9. RESET VIDEO PROCESSING
# ============================================================

def reset_video_processing(
    video_path,
    reason="reset"
):

    paths = ensure_video_output_dir(
        video_path
    )

    csv_path = (
        paths["detection_csv"]
    )

    state_path = (
        paths["state_json"]
    )

    # --------------------------------------------------------
    # Backup existing CSV
    # --------------------------------------------------------

    backup_path = (
        backup_detection_csv(
            video_path,
            reason
        )
    )

    # --------------------------------------------------------
    # Remove old CSV
    # --------------------------------------------------------

    if csv_path.exists():

        csv_path.unlink()

    # --------------------------------------------------------
    # Remove state
    # --------------------------------------------------------

    if state_path.exists():

        state_path.unlink()

    # --------------------------------------------------------
    # Create new CSV
    # --------------------------------------------------------

    empty_df = pd.DataFrame(
        columns=DETECTION_COLUMNS
    )

    temp_path = (
        csv_path.with_suffix(
            ".tmp.csv"
        )
    )

    empty_df.to_csv(

        temp_path,

        index=False,

        encoding="utf-8-sig"

    )

    temp_path.replace(
        csv_path
    )

    return {

        "backup":
            backup_path,

        "csv":
            csv_path,

        "state":
            state_path,

    }


# ============================================================
# 10. STATE WRITER
# ============================================================

def write_processing_state(
    video_path,
    status,
    next_frame_index,
    **extra
):

    paths = ensure_video_output_dir(
        video_path
    )

    state_path = (
        paths["state_json"]
    )

    payload = {

        "video_id":
            get_video_id(
                video_path
            ),

        "video_path":
            str(
                Path(video_path)
                .resolve()
            ),

        "status":
            status,

        "next_frame_index":
            int(
                next_frame_index
            ),

        "updated_at":
            utc_now_iso(),

        **extra,

    }

    temp_path = (
        state_path.with_suffix(
            ".tmp.json"
        )
    )

    temp_path.write_text(

        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False
        ),

        encoding="utf-8"

    )

    try:

        temp_path.replace(
            state_path
        )

    except PermissionError:

        time.sleep(
            0.2
        )

        if state_path.exists():

            try:

                state_path.unlink()

            except PermissionError:

                pass

        if temp_path.exists():

            temp_path.replace(
                state_path
            )


# ============================================================
# 11. READ STATE
# ============================================================

def read_processing_state(
    video_path
):

    paths = get_video_paths(
        video_path
    )

    state_path = (
        paths["state_json"]
    )

    if not state_path.exists():

        return None

    try:

        return json.loads(

            state_path.read_text(
                encoding="utf-8"
            )

        )

    except Exception:

        return None


# ============================================================
# 12. LOAD EXISTING CSV
# ============================================================

def load_existing_detection_rows(
    video_path
):

    paths = get_video_paths(
        video_path
    )

    csv_path = (
        paths["detection_csv"]
    )

    if not csv_path.exists():

        return pd.DataFrame(
            columns=DETECTION_COLUMNS
        )

    df = pd.read_csv(
        csv_path
    )

    # --------------------------------------------------------
    # Add missing columns
    #
    # IMPORTANT:
    # Old CSV without track_id is still usable.
    # --------------------------------------------------------

    for column in DETECTION_COLUMNS:

        if column not in df.columns:

            df[column] = None

    df = df[
        DETECTION_COLUMNS
    ]

    return df


# ============================================================
# 13. APPEND DETECTION ROWS
# ============================================================

def append_detection_rows(
    video_path,
    rows
):

    if not rows:

        return

    paths = get_video_paths(
        video_path
    )

    csv_path = (
        paths["detection_csv"]
    )

    # --------------------------------------------------------
    # New rows
    # --------------------------------------------------------

    df_new = pd.DataFrame(
        rows
    )

    for column in DETECTION_COLUMNS:

        if column not in df_new.columns:

            df_new[column] = None

    df_new = df_new[
        DETECTION_COLUMNS
    ]

    # --------------------------------------------------------
    # Existing rows
    # --------------------------------------------------------

    if csv_path.exists():

        df_old = pd.read_csv(
            csv_path
        )

        for column in DETECTION_COLUMNS:

            if column not in df_old.columns:

                df_old[column] = None

        df_old = df_old[
            DETECTION_COLUMNS
        ]

    else:

        df_old = pd.DataFrame(
            columns=DETECTION_COLUMNS
        )

    # --------------------------------------------------------
    # Duplicate protection
    # --------------------------------------------------------

    if len(df_old) > 0:

        existing_frames = set(

            pd.to_numeric(

                df_old[
                    "frame_index"
                ],

                errors="coerce"

            )
            .dropna()
            .astype(int)
            .tolist()

        )

        df_new = df_new[
            ~df_new[
                "frame_index"
            ]
            .astype(int)
            .isin(
                existing_frames
            )
        ]

    if len(df_new) == 0:

        return

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    df_final = pd.concat(

        [
            df_old,
            df_new
        ],

        ignore_index=True

    )

    # --------------------------------------------------------
    # Normalize frame index
    # --------------------------------------------------------

    df_final[
        "frame_index"
    ] = pd.to_numeric(

        df_final[
            "frame_index"
        ],

        errors="coerce"

    )

    df_final = (

        df_final

        .dropna(
            subset=[
                "frame_index"
            ]
        )

        .sort_values(
            "frame_index"
        )

        .reset_index(
            drop=True
        )

    )

    df_final[
        "frame_index"
    ] = (

        df_final[
            "frame_index"
        ]
        .astype(int)

    )

    # --------------------------------------------------------
    # Atomic write
    # --------------------------------------------------------

    temp_path = (
        csv_path.with_suffix(
            ".tmp.csv"
        )
    )

    df_final.to_csv(

        temp_path,

        index=False,

        encoding="utf-8-sig"

    )

    try:

        temp_path.replace(
            csv_path
        )

    except PermissionError:

        time.sleep(
            0.2
        )

        if csv_path.exists():

            try:

                csv_path.unlink()

            except PermissionError:

                pass

        if temp_path.exists():

            temp_path.replace(
                csv_path
            )


# ============================================================
# 14. OPEN VIDEO AT FRAME
# ============================================================

def open_video_at_frame(
    video_path,
    start_frame
):

    video_path = (
        Path(video_path)
        .resolve()
    )

    start_frame = int(
        start_frame
    )

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open video:\n"
            f"{video_path}"
        )

    if start_frame <= 0:

        return cap

    # --------------------------------------------------------
    # Try direct seek
    # --------------------------------------------------------

    try:

        cap.set(

            cv2.CAP_PROP_POS_FRAMES,

            start_frame

        )

        position = int(

            cap.get(
                cv2.CAP_PROP_POS_FRAMES
            )

        )

        if position == start_frame:

            return cap

    except Exception:

        pass

    # --------------------------------------------------------
    # Sequential fallback
    # --------------------------------------------------------

    cap.release()

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            "Could not reopen video."
        )

    print()
    print(
        f"Direct seek unavailable "
        f"for frame {start_frame}."
    )

    print(
        "Using sequential fallback..."
    )

    current_frame = 0

    while current_frame < start_frame:

        ok, _ = cap.read()

        if not ok:

            raise RuntimeError(

                f"Could not reach "
                f"resume frame "
                f"{start_frame}."

            )

        current_frame += 1

    return cap


# ============================================================
# 15. PROCESS VIDEO FRAMES
# ============================================================

def process_video_frames(
    video_path,
    start_frame=0
):

    video_path = (
        Path(video_path)
        .resolve()
    )

    video_id = get_video_id(
        video_path
    )

    paths = ensure_video_output_dir(
        video_path
    )

    # ========================================================
    # Video information
    # ========================================================

    info = get_video_info(
        video_path
    )

    total_frames = int(
        info["frame_count"]
    )

    fps = float(
        info["fps"]
    )

    expected_last_frame = (
        total_frames - 1
    )

    start_frame = max(
        0,
        int(start_frame)
    )

    if start_frame > expected_last_frame:

        raise RuntimeError(

            f"Invalid start frame: "
            f"{start_frame}\n"

            f"Expected range: "
            f"0-{expected_last_frame}"

        )

    print()
    print("=" * 70)
    print(
        f"PROCESSING : "
        f"{video_path.name}"
    )
    print(
        f"START FRAME: "
        f"{start_frame}"
    )
    print(
        f"LAST FRAME : "
        f"{expected_last_frame}"
    )
    print(
        f"TOTAL      : "
        f"{total_frames}"
    )
    print("=" * 70)

    # ========================================================
    # State
    # ========================================================

    write_processing_state(

        video_path,

        status="PROCESSING",

        next_frame_index=
            start_frame,

        total_frames=
            total_frames,

        last_expected_frame=
            expected_last_frame,

        skipped_frames=[],

    )

    # ========================================================
    # Open video
    # ========================================================

    cap = open_video_at_frame(

        video_path,

        start_frame

    )

    # ========================================================
    # YOLO
    # ========================================================

    model = YOLO(
        str(MODEL_PATH)
    )

    if torch.cuda.is_available():

        model.to(
            f"cuda:{DEVICE}"
        )

        print(
            f"GPU enabled: "
            f"CUDA:{DEVICE}"
        )

    else:

        print(
            "WARNING: CUDA unavailable."
        )

    # ========================================================
    # Progress
    # ========================================================

    frames_to_attempt = (

        expected_last_frame
        -
        start_frame
        +
        1

    )

    progress = tqdm(

        total=frames_to_attempt,

        desc=video_id,

        unit="frame"

    )

    pending_rows = []

    processed_frames = []

    skipped_frames = []

    current_frame = (
        start_frame
    )

    last_successful_frame = None

    try:

        while (
            current_frame
            <= expected_last_frame
        ):

            # =================================================
            # READ FRAME
            # =================================================

            ok, frame = cap.read()

            # -------------------------------------------------
            # CASE 1:
            # OpenCV failed to read frame
            # -------------------------------------------------

            if not ok or frame is None:

                print()
                print(
                    f"SKIP_FRAME: "
                    f"{current_frame}"
                )

                skipped_frames.append(
                    current_frame
                )

                write_processing_state(

                    video_path,

                    status="PROCESSING",

                    next_frame_index=
                        current_frame + 1,

                    total_frames=
                        total_frames,

                    last_expected_frame=
                        expected_last_frame,

                    last_successful_frame=
                        last_successful_frame,

                    skipped_frames=
                        skipped_frames,

                )

                progress.update(1)

                current_frame += 1

                continue

            # =================================================
            # TIMESTAMP
            # =================================================

            timestamp = (

                current_frame / fps

                if fps > 0

                else 0.0

            )

            # =================================================
            # YOLO
            # =================================================

            try:

                results = model.predict(

                    source=frame,

                    conf=CONFIDENCE_THRESHOLD,

                    iou=IOU_THRESHOLD,

                    device=DEVICE
                    if torch.cuda.is_available()
                    else "cpu",

                    verbose=False,

                )

            except Exception as detection_error:

                # ------------------------------------------------
                # Detection error:
                # We DO NOT skip silently.
                # Record frame and continue.
                # ------------------------------------------------

                print()
                print(
                    f"SKIP_FRAME: "
                    f"{current_frame} "
                    f"(detection error)"
                )

                print(
                    f"  {detection_error}"
                )

                skipped_frames.append(
                    current_frame
                )

                write_processing_state(

                    video_path,

                    status="PROCESSING",

                    next_frame_index=
                        current_frame + 1,

                    total_frames=
                        total_frames,

                    last_expected_frame=
                        expected_last_frame,

                    last_successful_frame=
                        last_successful_frame,

                    skipped_frames=
                        skipped_frames,

                )

                progress.update(1)

                current_frame += 1

                continue

            # =================================================
            # FACE DETECTION
            # =================================================

            face_detected = False

            track_id = None

            if results:

                result = results[0]

                if (

                    result.boxes
                    is not None

                    and

                    len(
                        result.boxes
                    ) > 0

                ):

                    face_detected = True

            # =================================================
            # ROW
            # =================================================

            row = {

                "video_id":
                    video_id,

                "frame_index":
                    current_frame,

                "timestamp":
                    timestamp,

                "face_detected":
                    face_detected,

                "track_id":
                    track_id,

            }

            pending_rows.append(
                row
            )

            processed_frames.append(
                current_frame
            )

            last_successful_frame = (
                current_frame
            )

            # =================================================
            # PERIODIC WRITE
            # =================================================

            if len(
                pending_rows
            ) >= 32:

                append_detection_rows(

                    video_path,

                    pending_rows

                )

                pending_rows = []

            # =================================================
            # STATE
            # =================================================

            write_processing_state(

                video_path,

                status="PROCESSING",

                next_frame_index=
                    current_frame + 1,

                total_frames=
                    total_frames,

                last_expected_frame=
                    expected_last_frame,

                last_successful_frame=
                    last_successful_frame,

                skipped_frames=
                    skipped_frames,

            )

            progress.update(1)

            current_frame += 1

        # =====================================================
        # FLUSH
        # =====================================================

        if pending_rows:

            append_detection_rows(

                video_path,

                pending_rows

            )

            pending_rows = []

        # =====================================================
        # FINAL CSV
        # =====================================================

        csv_path = (
            paths["detection_csv"]
        )

        if not csv_path.exists():

            raise RuntimeError(

                "Final detection.csv "
                "does not exist."

            )

        df_final = pd.read_csv(
            csv_path
        )

        if len(df_final) == 0:

            raise RuntimeError(
                "Final detection.csv "
                "is empty."
            )

        # =====================================================
        # VALID FRAME INDEX
        # =====================================================

        final_frames = (

            pd.to_numeric(

                df_final[
                    "frame_index"
                ],

                errors="coerce"

            )
            .dropna()
            .astype(int)

        )

        if len(final_frames) == 0:

            raise RuntimeError(

                "No valid frame_index "
                "in final CSV."

            )

        # =====================================================
        # DETECT MISSING FRAMES
        # =====================================================

        frame_set = set(
            final_frames.tolist()
        )

        missing_frames = [

            frame_index

            for frame_index
            in range(
                start_frame,
                expected_last_frame + 1
            )

            if (
                frame_index
                not in frame_set
            )

        ]

        # -----------------------------------------------------
        # Combine:
        # missing CSV frames + skipped frames
        # -----------------------------------------------------

        all_skipped_frames = sorted(

            set(
                skipped_frames
            )
            |
            set(
                missing_frames
            )

        )

        # =====================================================
        # RESULT
        # =====================================================

        if len(
            all_skipped_frames
        ) == 0:

            final_status = (
                "COMPLETED"
            )

        else:

            final_status = (
                "COMPLETED_WITH_SKIPS"
            )

        # =====================================================
        # FINAL STATE
        # =====================================================

        write_processing_state(

            video_path,

            status=final_status,

            next_frame_index=
                total_frames,

            total_frames=
                total_frames,

            last_expected_frame=
                expected_last_frame,

            last_successful_frame=
                last_successful_frame,

            processed_frames=
                len(
                    final_frames
                ),

            skipped_count=
                len(
                    all_skipped_frames
                ),

            skipped_frames=
                all_skipped_frames,

            completed_at=
                utc_now_iso(),

        )

        # =====================================================
        # SUMMARY
        # =====================================================

        print()
        print("=" * 70)
        print(
            f"PROCESSING FINISHED: "
            f"{video_id}"
        )
        print("=" * 70)

        print(
            f"Status            : "
            f"{final_status}"
        )

        print(
            f"Expected frames   : "
            f"{total_frames}"
        )

        print(
            f"CSV rows          : "
            f"{len(df_final)}"
        )

        print(
            f"Processed frames  : "
            f"{len(final_frames)}"
        )

        print(
            f"Skipped frames    : "
            f"{len(all_skipped_frames)}"
        )

        if all_skipped_frames:

            print()
            print(
                "Skipped frame list:"
            )

            print(
                all_skipped_frames
            )

        print("=" * 70)

        return {

            "status":
                final_status,

            "video_id":
                video_id,

            "total_frames":
                total_frames,

            "csv_rows":
                len(df_final),

            "processed_frames":
                len(final_frames),

            "skipped_frames":
                all_skipped_frames,

            "skipped_count":
                len(
                    all_skipped_frames
                ),

        }

    except Exception:

        # =====================================================
        # REAL FAILURE
        # =====================================================

        write_processing_state(

            video_path,

            status="FAILED",

            next_frame_index=
                current_frame,

            total_frames=
                total_frames,

            last_expected_frame=
                expected_last_frame,

            last_successful_frame=
                last_successful_frame,

            skipped_frames=
                skipped_frames,

            failed_at=
                utc_now_iso(),

        )

        raise

    finally:

        progress.close()

        cap.release()

        del model

        try:

            torch.cuda.empty_cache()

        except Exception:

            pass

        gc.collect()


# ============================================================
# CELL 7 READY
# ============================================================

print()
print("=" * 70)
print("PHASE 2 — CELL 7 READY")
print("=" * 70)

print()
print("Processing engine loaded:")
print("  [OK] reset_video_processing()")
print("  [OK] backup_detection_csv()")
print("  [OK] write_processing_state()")
print("  [OK] read_processing_state()")
print("  [OK] open_video_at_frame()")
print("  [OK] append_detection_rows()")
print("  [OK] process_video_frames()")
print("  [OK] SKIP_FRAME")
print("  [OK] COMPLETED_WITH_SKIPS")
print("  [OK] .webm support")
print("  [OK] CUDA support")
print("  [OK] duplicate protection")
print("  [OK] final validation")
print("=" * 70)


PHASE 2 — CELL 7 READY

Processing engine loaded:
  [OK] reset_video_processing()
  [OK] backup_detection_csv()
  [OK] write_processing_state()
  [OK] read_processing_state()
  [OK] open_video_at_frame()
  [OK] append_detection_rows()
  [OK] process_video_frames()
  [OK] SKIP_FRAME
  [OK] COMPLETED_WITH_SKIPS
  [OK] .webm support
  [OK] CUDA support
  [OK] duplicate protection
  [OK] final validation


In [10]:
# ============================================================
# PHASE 2 — CELL 8
# Production Batch Runner
#
# Compatible with:
#   Cell 7 — SKIP_FRAME engine
#
# Behavior:
#   RUN
#   RESET
#   RESUME
#   SKIP
#   COMPLETED
#   COMPLETED_WITH_SKIPS
#
# IMPORTANT:
#   Missing frames are allowed.
#   Cell 8 does NOT require every frame to exist.
# ============================================================

import traceback
import gc
from pathlib import Path

import pandas as pd
import torch


# ============================================================
# 1. DISCOVER VIDEOS
# ============================================================

def discover_videos():

    videos = []

    if not RAW_VIDEO_DIR.exists():

        raise FileNotFoundError(
            f"Raw video directory not found:\n"
            f"{RAW_VIDEO_DIR}"
        )

    for path in RAW_VIDEO_DIR.iterdir():

        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in SUPPORTED_VIDEO_EXTENSIONS
        ):
            continue

        videos.append(
            path.resolve()
        )

    videos.sort(
        key=lambda p: p.name.lower()
    )

    return videos


# ============================================================
# 2. FINAL CSV VALIDATION
#
# IMPORTANT:
#   This validation DOES NOT require the last frame.
#
#   Missing frames are allowed because Cell 7 uses
#   SKIP_FRAME.
# ============================================================

def validate_final_detection_csv(
    video_path
):

    paths = get_video_paths(
        video_path
    )

    csv_path = (
        paths["detection_csv"]
    )

    result = {

        "valid": False,

        "reason": None,

        "rows": 0,

        "expected_frames": 0,

        "processed_frames": 0,

        "missing_frames": 0,

        "skipped_frames": [],

    }

    # ========================================================
    # CSV exists
    # ========================================================

    if not csv_path.exists():

        result["reason"] = (
            "detection_csv_missing"
        )

        return result

    # ========================================================
    # Read CSV
    # ========================================================

    try:

        df = pd.read_csv(
            csv_path
        )

    except Exception as e:

        result["reason"] = (
            f"csv_read_error:{e}"
        )

        return result

    # ========================================================
    # Required columns
    # ========================================================

    required_columns = [
        "video_id",
        "frame_index",
        "timestamp",
        "face_detected",
        "track_id",
    ]

    missing_columns = [

        column

        for column
        in required_columns

        if column
        not in df.columns

    ]

    # --------------------------------------------------------
    # Old CSV may not contain track_id.
    #
    # Add it instead of failing.
    # --------------------------------------------------------

    if "track_id" not in df.columns:

        df["track_id"] = None

        try:

            temp_path = (
                csv_path.with_suffix(
                    ".tmp.csv"
                )
            )

            df.to_csv(
                temp_path,
                index=False,
                encoding="utf-8-sig"
            )

            temp_path.replace(
                csv_path
            )

        except Exception:

            pass

        missing_columns = [

            column

            for column
            in missing_columns

            if column != "track_id"

        ]

    if missing_columns:

        result["reason"] = (

            "missing_columns:"
            +
            ",".join(
                missing_columns
            )

        )

        return result

    # ========================================================
    # Empty CSV
    # ========================================================

    if len(df) == 0:

        result["reason"] = (
            "csv_empty"
        )

        return result

    # ========================================================
    # Frame index
    # ========================================================

    df["frame_index"] = pd.to_numeric(

        df["frame_index"],

        errors="coerce"

    )

    df = df.dropna(
        subset=[
            "frame_index"
        ]
    )

    if len(df) == 0:

        result["reason"] = (
            "no_valid_frame_index"
        )

        return result

    df["frame_index"] = (
        df["frame_index"]
        .astype(int)
    )

    # ========================================================
    # Video info
    # ========================================================

    try:

        video_info = get_video_info(
            video_path
        )

    except Exception as e:

        result["reason"] = (
            f"video_info_error:{e}"
        )

        return result

    total_frames = int(
        video_info["frame_count"]
    )

    expected_last_frame = (
        total_frames - 1
    )

    # ========================================================
    # Unique frames
    # ========================================================

    frame_indices = sorted(

        set(
            df[
                "frame_index"
            ].tolist()
        )

    )

    # ========================================================
    # Out-of-range frames
    # ========================================================

    invalid_frames = [

        frame

        for frame
        in frame_indices

        if (
            frame < 0
            or
            frame > expected_last_frame
        )

    ]

    if invalid_frames:

        result["reason"] = (

            "invalid_frame_indices:"
            +
            str(
                invalid_frames[:10]
            )

        )

        return result

    # ========================================================
    # Missing frames
    #
    # Missing frames are NOT an error.
    # ========================================================

    frame_set = set(
        frame_indices
    )

    missing_frames = [

        frame

        for frame
        in range(
            total_frames
        )

        if frame
        not in frame_set

    ]

    # ========================================================
    # State
    # ========================================================

    state = read_processing_state(
        video_path
    )

    if state:

        skipped_frames = (

            state.get(
                "skipped_frames",
                []
            )

        )

    else:

        skipped_frames = []

    # ========================================================
    # Combine missing + skipped
    # ========================================================

    all_skipped_frames = sorted(

        set(
            int(frame)
            for frame
            in skipped_frames
        )
        |
        set(
            missing_frames
        )

    )

    # ========================================================
    # Validation
    #
    # CSV is valid as long as:
    #   - CSV exists
    #   - schema is valid
    #   - at least one frame exists
    #   - frame indices are valid
    #
    # Missing frames are allowed.
    # ========================================================

    result["valid"] = True

    result["rows"] = int(
        len(df)
    )

    result["expected_frames"] = (
        total_frames
    )

    result["processed_frames"] = (
        len(frame_set)
    )

    result["missing_frames"] = (
        len(missing_frames)
    )

    result["skipped_frames"] = (
        all_skipped_frames
    )

    if all_skipped_frames:

        result["reason"] = (

            "valid_with_skips:"
            +
            str(
                len(
                    all_skipped_frames
                )
            )

        )

    else:

        result["reason"] = (
            "valid_complete"
        )

    return result


# ============================================================
# 3. BUILD PROCESSING PLAN
# ============================================================

def build_processing_plan(
    videos
):

    plan = []

    print()
    print("=" * 70)
    print(
        "PHASE 2 — SAFE PROCESSING PLAN"
    )
    print("=" * 70)

    for index, video_path in enumerate(
        videos,
        start=1
    ):

        decision = (
            determine_processing_action(
                video_path
            )
        )

        item = {

            "index":
                index,

            "video_path":
                video_path,

            "video_id":
                get_video_id(
                    video_path
                ),

            "action":
                decision[
                    "action"
                ],

            "reason":
                decision[
                    "reason"
                ],

            "start_frame":
                decision[
                    "start_frame"
                ],

            "validation":
                decision.get(
                    "validation"
                ),

        }

        plan.append(
            item
        )

        print()

        print(
            f"[{index}/{len(videos)}] "
            f"{video_path.name}"
        )

        print(
            f"Action      : "
            f"{item['action']}"
        )

        print(
            f"Reason      : "
            f"{item['reason']}"
        )

        if (
            item["start_frame"]
            is not None
        ):

            print(
                f"Start frame : "
                f"{item['start_frame']}"
            )

    print()
    print("=" * 70)

    return plan


# ============================================================
# 4. RUN ONE VIDEO
# ============================================================

def run_one_video(item):

    video_path = (
        item["video_path"]
    )

    video_id = (
        item["video_id"]
    )

    action = (
        item["action"]
    )

    reason = (
        item["reason"]
    )

    start_frame = (
        item["start_frame"]
    )

    # ========================================================
    # SKIP
    # ========================================================

    if action == "SKIP":

        print()
        print(
            f"SKIP: {video_id}"
        )

        print(
            f"Reason: {reason}"
        )

        return {

            "video_id":
                video_id,

            "status":
                "SKIPPED",

            "action":
                action,

            "reason":
                reason,

        }

    # ========================================================
    # LOCK
    # ========================================================

    with VideoProcessLock(
        video_path
    ):

        print()
        print("=" * 70)

        print(
            f"PROCESSING : "
            f"{video_path.name}"
        )

        print(
            f"ACTION     : "
            f"{action}"
        )

        print(
            f"REASON     : "
            f"{reason}"
        )

        print(
            f"START FRAME: "
            f"{start_frame}"
        )

        print("=" * 70)

        try:

            # =================================================
            # RESET
            # =================================================

            if action == "RESET":

                print()
                print(
                    "Resetting processing..."
                )

                reset_result = (
                    reset_video_processing(
                        video_path,
                        reason
                    )
                )

                if (
                    reset_result
                    .get("backup")
                ):

                    print(
                        "Backup created:"
                    )

                    print(
                        f"  "
                        f"{reset_result['backup']}"
                    )

                start_frame = 0

            # =================================================
            # RUN
            # =================================================

            elif action == "RUN":

                start_frame = 0

            # =================================================
            # RESUME
            # =================================================

            elif action == "RESUME":

                if start_frame is None:

                    start_frame = 0

                print()
                print(
                    "Resuming from "
                    f"frame {start_frame}"
                )

            else:

                raise RuntimeError(

                    f"Unknown action: "
                    f"{action}"

                )

            # =================================================
            # PROCESS
            # =================================================

            process_result = (
                process_video_frames(

                    video_path,

                    start_frame=
                        start_frame

                )
            )

            # =================================================
            # FINAL VALIDATION
            #
            # SKIP_FRAME aware.
            # =================================================

            final_validation = (
                validate_final_detection_csv(
                    video_path
                )
            )

            if not final_validation[
                "valid"
            ]:

                raise RuntimeError(

                    "Final CSV validation "
                    "failed: "
                    +
                    str(
                        final_validation[
                            "reason"
                        ]
                    )

                )

            # =================================================
            # STATUS FROM CELL 7
            # =================================================

            process_status = (
                process_result.get(
                    "status",
                    "COMPLETED"
                )
            )

            if process_status == (
                "COMPLETED_WITH_SKIPS"
            ):

                final_status = (
                    "COMPLETED_WITH_SKIPS"
                )

            elif (
                final_validation[
                    "skipped_frames"
                ]
            ):

                final_status = (
                    "COMPLETED_WITH_SKIPS"
                )

            else:

                final_status = (
                    "COMPLETED"
                )

            # =================================================
            # PRINT FINAL RESULT
            # =================================================

            print()
            print("=" * 70)

            print(
                f"FINISHED: "
                f"{video_id}"
            )

            print(
                f"Status            : "
                f"{final_status}"
            )

            print(
                f"Expected frames   : "
                f"{final_validation['expected_frames']}"
            )

            print(
                f"CSV rows          : "
                f"{final_validation['rows']}"
            )

            print(
                f"Processed frames  : "
                f"{final_validation['processed_frames']}"
            )

            print(
                f"Skipped frames    : "
                f"{final_validation['missing_frames']}"
            )

            if final_validation[
                "skipped_frames"
            ]:

                print()
                print(
                    "Skipped frame list:"
                )

                print(
                    final_validation[
                        "skipped_frames"
                    ]
                )

            print("=" * 70)

            return {

                "video_id":
                    video_id,

                "status":
                    final_status,

                "action":
                    action,

                "reason":
                    reason,

                "rows":
                    final_validation[
                        "rows"
                    ],

                "expected_frames":
                    final_validation[
                        "expected_frames"
                    ],

                "processed_frames":
                    final_validation[
                        "processed_frames"
                    ],

                "skipped_frames":
                    final_validation[
                        "skipped_frames"
                    ],

                "skipped_count":
                    len(
                        final_validation[
                            "skipped_frames"
                        ]
                    ),

            }

        except Exception as e:

            error_text = (
                f"{type(e).__name__}: "
                f"{e}"
            )

            print()
            print(
                f"FAILED: {video_id}"
            )

            print(
                f"Error: {error_text}"
            )

            traceback.print_exc()

            return {

                "video_id":
                    video_id,

                "status":
                    "FAILED",

                "action":
                    action,

                "reason":
                    reason,

                "error":
                    error_text,

            }

        finally:

            try:

                torch.cuda.empty_cache()

            except Exception:

                pass

            gc.collect()


# ============================================================
# 5. DISCOVER VIDEOS
# ============================================================

VIDEOS = discover_videos()

if not VIDEOS:

    raise RuntimeError(
        "No supported videos found."
    )


print()
print(
    f"Videos found: "
    f"{len(VIDEOS)}"
)


# ============================================================
# 6. BUILD PROCESSING PLAN
# ============================================================

PROCESSING_PLAN = (
    build_processing_plan(
        VIDEOS
    )
)


# ============================================================
# 7. EXECUTE BATCH
# ============================================================

BATCH_RESULTS = []


for item in PROCESSING_PLAN:

    result = run_one_video(
        item
    )

    BATCH_RESULTS.append(
        result
    )


# ============================================================
# 8. BATCH SUMMARY
# ============================================================

print()
print("=" * 70)
print(
    "PHASE 2 — BATCH SUMMARY"
)
print("=" * 70)

for result in BATCH_RESULTS:

    print(

        f"{result['video_id']:<15}"
        f"{result['status']:<24}"
        f"{result.get('action', '')}"

    )

print("=" * 70)


# ============================================================
# 9. FINAL COUNTS
# ============================================================

completed = sum(

    1

    for r in BATCH_RESULTS

    if r["status"]
    == "COMPLETED"

)

completed_with_skips = sum(

    1

    for r in BATCH_RESULTS

    if r["status"]
    == "COMPLETED_WITH_SKIPS"

)

failed = sum(

    1

    for r in BATCH_RESULTS

    if r["status"]
    == "FAILED"

)

skipped = sum(

    1

    for r in BATCH_RESULTS

    if r["status"]
    == "SKIPPED"

)


print()
print("=" * 70)
print("FINAL BATCH RESULT")
print("=" * 70)

print(
    f"COMPLETED             : "
    f"{completed}"
)

print(
    f"COMPLETED_WITH_SKIPS  : "
    f"{completed_with_skips}"
)

print(
    f"FAILED                : "
    f"{failed}"
)

print(
    f"SKIPPED               : "
    f"{skipped}"
)

print("=" * 70)


Videos found: 3

PHASE 2 — SAFE PROCESSING PLAN

[1/3] video001.webm
Action      : RESUME
Reason      : missing_tail_frames:43725-43725
Start frame : 43725

[2/3] video002.webm
Action      : RESUME
Reason      : missing_tail_frames:256-39500
Start frame : 256

[3/3] video003.webm
Action      : RESET
Reason      : missing_columns:track_id
Start frame : 0


PROCESSING : video001.webm
ACTION     : RESUME
REASON     : missing_tail_frames:43725-43725
START FRAME: 43725

Resuming from frame 43725

PROCESSING : video001.webm
START FRAME: 43725
LAST FRAME : 43725
TOTAL      : 43726
GPU enabled: CUDA:0


video001:   0%|          | 0/1 [00:00<?, ?frame/s]


SKIP_FRAME: 43725

PROCESSING FINISHED: video001
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 43726
CSV rows          : 43725
Processed frames  : 43725
Skipped frames    : 1

Skipped frame list:
[43725]

FINISHED: video001
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 43726
CSV rows          : 43725
Processed frames  : 43725
Skipped frames    : 1

Skipped frame list:
[43725]

PROCESSING : video002.webm
ACTION     : RESUME
REASON     : missing_tail_frames:256-39500
START FRAME: 256

Resuming from frame 256

PROCESSING : video002.webm
START FRAME: 256
LAST FRAME : 39500
TOTAL      : 39501
GPU enabled: CUDA:0


video002:   0%|          | 0/39245 [00:00<?, ?frame/s]


SKIP_FRAME: 39500

PROCESSING FINISHED: video002
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 39501
CSV rows          : 39500
Processed frames  : 39500
Skipped frames    : 1

Skipped frame list:
[39500]

FINISHED: video002
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 39501
CSV rows          : 39500
Processed frames  : 39500
Skipped frames    : 1

Skipped frame list:
[39500]

PROCESSING : video003.webm
ACTION     : RESET
REASON     : missing_columns:track_id
START FRAME: 0

Resetting processing...

Backup created:
  C:\LipReadingSSL\output\video003\backups\detection_20260817_222115_missing_columns_track_id.csv
Backup created:
  C:\LipReadingSSL\output\video003\backups\detection_20260817_222115_missing_columns_track_id.csv

PROCESSING : video003.webm
START FRAME: 0
LAST FRAME : 42624
TOTAL      : 42625
GPU enabled: CUDA:0


video003:   0%|          | 0/42625 [00:00<?, ?frame/s]

C:\Users\User\AppData\Local\Temp\ipykernel_29212\612835767.py:748: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(



SKIP_FRAME: 42624

PROCESSING FINISHED: video003
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 42625
CSV rows          : 42624
Processed frames  : 42624
Skipped frames    : 1

Skipped frame list:
[42624]

FINISHED: video003
Status            : COMPLETED_WITH_SKIPS
Expected frames   : 42625
CSV rows          : 42624
Processed frames  : 42624
Skipped frames    : 1

Skipped frame list:
[42624]

PHASE 2 — BATCH SUMMARY
video001       COMPLETED_WITH_SKIPS    RESUME
video002       COMPLETED_WITH_SKIPS    RESUME
video003       COMPLETED_WITH_SKIPS    RESET

FINAL BATCH RESULT
COMPLETED             : 0
COMPLETED_WITH_SKIPS  : 3
FAILED                : 0
SKIPPED               : 0


In [13]:
# ============================================================
# PHASE 2 — CELL 9
# Final Batch Validation + Summary
#
# Compatible with:
#   Cell 6 — Production Validation
#   Cell 7 — SKIP_FRAME
#   Cell 8 — Batch Runner
#
# IMPORTANT:
#   - Does NOT use FINAL_VALIDATION_DF
#   - Uses BATCH_RESULTS from Cell 8
#   - Re-validates every video after processing
# ============================================================

import pandas as pd
from pathlib import Path


# ============================================================
# Final Validation
# ============================================================

def build_final_validation(
    videos
):

    results = []

    print()
    print("=" * 70)
    print("PHASE 2 — FINAL VALIDATION")
    print("=" * 70)

    for video_path in videos:

        video_path = (
            Path(video_path)
            .resolve()
        )

        video_id = get_video_id(
            video_path
        )

        # ----------------------------------------------------
        # Validate detection CSV
        # ----------------------------------------------------

        validation = (
            validate_detection_csv(
                video_path
            )
        )

        # ----------------------------------------------------
        # Get state
        # ----------------------------------------------------

        try:

            state = read_processing_state(
                video_path
            )

            if not isinstance(
                state,
                dict
            ):

                state = {}

        except Exception:

            state = {}

        # ----------------------------------------------------
        # Skipped frames
        # ----------------------------------------------------

        skipped_frames = (
            state.get(
                "skipped_frames",
                []
            )
        )

        if not isinstance(
            skipped_frames,
            (list, tuple, set)
        ):

            skipped_frames = []

        skipped_frames = [

            int(frame)

            for frame
            in skipped_frames

            if str(frame).isdigit()

        ]

        skipped_frames = sorted(
            set(
                skipped_frames
            )
        )

        # ----------------------------------------------------
        # CSV path
        # ----------------------------------------------------

        paths = get_video_paths(
            video_path
        )

        csv_path = Path(
            paths[
                "detection_csv"
            ]
        )

        # ----------------------------------------------------
        # Determine final status
        # ----------------------------------------------------

        if validation["valid"]:

            if validation["reason"] == (
                "valid_with_skips"
            ):

                final_status = (
                    "COMPLETED_WITH_SKIPS"
                )

            else:

                final_status = (
                    "COMPLETED"
                )

            final_ok = True

        else:

            final_status = (
                "INCOMPLETE"
            )

            final_ok = False

        # ----------------------------------------------------
        # Build result
        # ----------------------------------------------------

        item = {

            "video_id":
                video_id,

            "video":
                video_path.name,

            "csv_exists":
                csv_path.exists(),

            "csv_rows":
                validation[
                    "rows"
                ],

            "first_frame":
                validation[
                    "first_frame"
                ],

            "last_frame":
                validation[
                    "last_frame"
                ],

            "expected_last_frame":
                validation[
                    "expected_last_frame"
                ],

            "missing_frames":
                len(
                    validation[
                        "missing_frames"
                    ]
                ),

            "skipped_frames":
                len(
                    skipped_frames
                ),

            "unresolved_frames":
                len(
                    validation[
                        "unresolved_frames"
                    ]
                ),

            "schema_ok":
                len(
                    validation[
                        "missing_columns"
                    ]
                ) == 0,

            "final_ok":
                final_ok,

            "status":
                final_status,

            "reason":
                validation[
                    "reason"
                ],

        }

        results.append(
            item
        )

        # ----------------------------------------------------
        # Print
        # ----------------------------------------------------

        print()
        print(
            f"{video_id}"
        )

        print(
            f"  Status          : "
            f"{final_status}"
        )

        print(
            f"  CSV             : "
            f"{csv_path}"
        )

        print(
            f"  CSV exists      : "
            f"{item['csv_exists']}"
        )

        print(
            f"  CSV rows        : "
            f"{item['csv_rows']}"
        )

        print(
            f"  Last frame      : "
            f"{item['last_frame']}"
        )

        print(
            f"  Expected last   : "
            f"{item['expected_last_frame']}"
        )

        print(
            f"  Missing frames  : "
            f"{item['missing_frames']}"
        )

        print(
            f"  Skipped frames  : "
            f"{item['skipped_frames']}"
        )

        print(
            f"  Unresolved      : "
            f"{item['unresolved_frames']}"
        )

        print(
            f"  Schema OK       : "
            f"{item['schema_ok']}"
        )

        print(
            f"  Final OK        : "
            f"{item['final_ok']}"
        )

        print(
            f"  Reason          : "
            f"{item['reason']}"
        )

    print()
    print("=" * 70)

    return results


# ============================================================
# Run Final Validation
# ============================================================

FINAL_VALIDATION_RESULTS = (
    build_final_validation(
        VIDEOS
    )
)


# ============================================================
# Convert to DataFrame
# ============================================================

FINAL_VALIDATION_TABLE = (
    pd.DataFrame(
        FINAL_VALIDATION_RESULTS
    )
)


# ============================================================
# Final Summary
# ============================================================

print()
print("=" * 70)
print("PHASE 2 — FINAL SUMMARY")
print("=" * 70)


if FINAL_VALIDATION_TABLE.empty:

    print(
        "No validation results."
    )

else:

    for result in (
        FINAL_VALIDATION_RESULTS
    ):

        print(
            f"{result['video_id']:<15}"
            f"{result['status']:<25}"
            f"rows={result['csv_rows']:<8}"
            f"skipped={result['skipped_frames']:<6}"
        )


# ============================================================
# Statistics
# ============================================================

total_videos = len(
    FINAL_VALIDATION_RESULTS
)

completed_videos = sum(

    1

    for result
    in FINAL_VALIDATION_RESULTS

    if result["status"] == "COMPLETED"

)

completed_with_skips = sum(

    1

    for result
    in FINAL_VALIDATION_RESULTS

    if result["status"]
    == "COMPLETED_WITH_SKIPS"

)

incomplete_videos = sum(

    1

    for result
    in FINAL_VALIDATION_RESULTS

    if result["status"]
    == "INCOMPLETE"

)


# ============================================================
# Overall Result
# ============================================================

all_ok = (

    incomplete_videos == 0

)


print()
print("=" * 70)

print(
    f"Total videos           : "
    f"{total_videos}"
)

print(
    f"Completed              : "
    f"{completed_videos}"
)

print(
    f"Completed with skips   : "
    f"{completed_with_skips}"
)

print(
    f"Incomplete             : "
    f"{incomplete_videos}"
)

print(
    f"Overall validation     : "
    f"{'PASS' if all_ok else 'FAIL'}"
)

print("=" * 70)


# ============================================================
# Optional: Display DataFrame
# ============================================================

try:

    display(
        FINAL_VALIDATION_TABLE
    )

except Exception:

    print()
    print(
        FINAL_VALIDATION_TABLE
    )


PHASE 2 — FINAL VALIDATION

video001
  Status          : COMPLETED_WITH_SKIPS
  CSV             : C:\LipReadingSSL\output\video001\detection.csv
  CSV exists      : True
  CSV rows        : 43725
  Last frame      : 43724
  Expected last   : 43725
  Missing frames  : 1
  Skipped frames  : 1
  Unresolved      : 0
  Schema OK       : True
  Final OK        : True
  Reason          : valid_with_skips

video002
  Status          : COMPLETED_WITH_SKIPS
  CSV             : C:\LipReadingSSL\output\video002\detection.csv
  CSV exists      : True
  CSV rows        : 39500
  Last frame      : 39499
  Expected last   : 39500
  Missing frames  : 1
  Skipped frames  : 1
  Unresolved      : 0
  Schema OK       : True
  Final OK        : True
  Reason          : valid_with_skips

video003
  Status          : COMPLETED_WITH_SKIPS
  CSV             : C:\LipReadingSSL\output\video003\detection.csv
  CSV exists      : True
  CSV rows        : 42624
  Last frame      : 42623
  Expected last   : 42624
  M

,video_id,video,csv_exists,csv_rows,first_frame,last_frame,expected_last_frame,missing_frames,skipped_frames,unresolved_frames,schema_ok,final_ok,status,reason
0,video001,video001.webm,True,43725,0,43724,43725,1,1,0,True,True,COMPLETED_WITH_SKIPS,valid_with_skips
1,video002,video002.webm,True,39500,0,39499,39500,1,1,0,True,True,COMPLETED_WITH_SKIPS,valid_with_skips
2,video003,video003.webm,True,42624,0,42623,42624,1,1,0,True,True,COMPLETED_WITH_SKIPS,valid_with_skips
